# `select` — Reference

`select` picks columns by name, slice, dtype, or substring. Negative names (`'-col'`, `'~col'`), negative slices (`'-start:end'`), and `exclude=` drop columns while preserving relative order. Regex is the `regex=` keyword — a positional string that is not an exact name is an error, not a silent regex. `everything()` pulls in "the rest" after pinning specific columns first.

| Argument type | Example | Selects |
|---|---|---|
| List | `['a','b']` | exact named columns |
| String (name) | `'species'` | exact column |
| Negative string | `'-species'`, `'~island'` | exclude named column |
| Negative slice | `'-bill_length_mm:body_mass_g'` | exclude contiguous range |
| `exclude=` | `exclude='species'`, `exclude=['a','b']` | exclude specific column(s) |
| String (slice) | `'bill_length_mm:body_mass_g'` | contiguous range |
| `everything()` | after explicit cols | remaining columns |
| callable | `lambda c: c.startswith('b')` | columns passing test |
| `regex=` | `regex='mm$'` | names matching regex |
| `dtype=` | `dtype='numeric'` | by dtype |
| `exclude_dtype=` | `exclude_dtype='number'` | excluding dtype |
| `contains=` | `contains='mm'` | substring in name |
| `startswith=` | `startswith='bill'` | name prefix |
| `endswith=` | `endswith='mm'` | name suffix |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']
tips = pt.sample_data['tips']

penguins.cols(ascending=None)  # see column order

['species',
 'island',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex']

## Names, `contains`, and `regex`
Criteria are a union (OR). Prefer `contains=` for substrings and `regex=` for patterns. Multiple values in a list are also OR.

In [2]:
# All columns that contain 'bill' OR 'mass', plus species
(penguins
 .select('species', contains=['bill', 'mass'])
 .head()
)

species,bill_length_mm,bill_depth_mm,body_mass_g
str,f64,f64,f64
"""Adelie""",39.1,18.7,3750.0
"""Adelie""",39.5,17.4,3800.0
"""Adelie""",40.3,18.0,3250.0
"""Adelie""",null,null,null
"""Adelie""",36.7,19.3,3450.0


In [3]:
# Column order rule: explicit list preserves YOUR order; contains/regex follow original DataFrame order
# island, sex, species are listed explicitly so they come first in that exact order,
# then contains='mm' adds remaining mm-cols in original df order
(penguins
 .select('island', 'sex', 'species', contains='mm')
 .head()
)

island,sex,species,bill_length_mm,bill_depth_mm,flipper_length_mm
str,str,str,f64,f64,f64
"""Torgersen""","""Male""","""Adelie""",39.1,18.7,181.0
"""Torgersen""","""Female""","""Adelie""",39.5,17.4,186.0
"""Torgersen""","""Female""","""Adelie""",40.3,18.0,195.0
"""Torgersen""",null,"""Adelie""",null,null,null
"""Torgersen""","""Female""","""Adelie""",36.7,19.3,193.0


In [4]:
# Columns ending in 'mm' — anchored regex
(penguins
 .select(regex='mm$')
 .head()
)

bill_length_mm,bill_depth_mm,flipper_length_mm
f64,f64,f64
39.1,18.7,181.0
39.5,17.4,186.0
40.3,18.0,195.0
null,null,null
36.7,19.3,193.0


In [5]:
# Combining contains= with a slice in one call (union)
# contains='mm' picks the mm-cols; 'island:body_mass_g' slice picks everything between those two cols
(penguins
 .select('island:body_mass_g', contains='mm')
 .head()
)

island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
str,f64,f64,f64,f64
"""Torgersen""",39.1,18.7,181.0,3750.0
"""Torgersen""",39.5,17.4,186.0,3800.0
"""Torgersen""",40.3,18.0,195.0,3250.0
"""Torgersen""",null,null,null,null
"""Torgersen""",36.7,19.3,193.0,3450.0


## Slice notation — contiguous column range
`'start_col:end_col'` selects all columns from start to end (inclusive), in DataFrame order.

In [6]:
# Select from 'bill_length_mm' through 'body_mass_g' (all measurement cols)
(penguins
 .select('bill_length_mm:body_mass_g')
 .head()
)

bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
f64,f64,f64,f64
39.1,18.7,181.0,3750.0
39.5,17.4,186.0,3800.0
40.3,18.0,195.0,3250.0
null,null,null,null
36.7,19.3,193.0,3450.0


In [7]:
# Advanced regex: columns containing 'mm' but NOT 'flipper'
# this is regex for selecting all columns containing 'mm' but not 'flipper'.
# just because it can be done, does not mean it should be done.
# use more readable ways like list comprehension — this will ensure your co-workers
# don't stab you in the middle of the night
(penguins
 .select(regex=r'^(?=.*mm)(?!.*flipper).*')  # regex sorcery
 .head()
)

# More readable alternative:
cols_with_mm_not_flipper = [c for c in penguins.columns if 'mm' in c and 'flipper' not in c]
(
    penguins
    .select(cols_with_mm_not_flipper)
    .head()
)


bill_length_mm,bill_depth_mm
f64,f64
39.1,18.7
39.5,17.4
40.3,18.0
null,null
36.7,19.3


## `everything()` — pin key columns first, then the rest
Use `everything()` to move specific columns to the front without dropping any.

In [8]:
# Move 'body_mass_g' and 'species' to front, keep all other columns after
(penguins
 .select('body_mass_g', 'species', pt.everything())
 .head()
)

body_mass_g,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,sex
f64,str,str,f64,f64,f64,str
3750.0,"""Adelie""","""Torgersen""",39.1,18.7,181.0,"""Male"""
3800.0,"""Adelie""","""Torgersen""",39.5,17.4,186.0,"""Female"""
3250.0,"""Adelie""","""Torgersen""",40.3,18.0,195.0,"""Female"""
null,"""Adelie""","""Torgersen""",null,null,null,null
3450.0,"""Adelie""","""Torgersen""",36.7,19.3,193.0,"""Female"""


## Callable — select columns with a function

In [9]:
# Lambda: columns whose name has more than 6 characters
(penguins
 .select(lambda c: len(c) > 6)
 .head()
)

species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
str,f64,f64,f64,f64
"""Adelie""",39.1,18.7,181.0,3750.0
"""Adelie""",39.5,17.4,186.0,3800.0
"""Adelie""",40.3,18.0,195.0,3250.0
"""Adelie""",null,null,null,null
"""Adelie""",36.7,19.3,193.0,3450.0


## `dtype`, `exclude_dtype`, `contains`, `startswith`, `endswith`, `regex`

In [10]:
# All numeric columns only
(penguins
 .select(dtype='numeric')
 .head()
)

bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
f64,f64,f64,f64
39.1,18.7,181.0,3750.0
39.5,17.4,186.0,3800.0
40.3,18.0,195.0,3250.0
null,null,null,null
36.7,19.3,193.0,3450.0


In [11]:
# Drop numeric columns — keep only categorical/string columns
(penguins
 .select(exclude_dtype='number')
 .head()
)

species,island,sex
str,str,str
"""Adelie""","""Torgersen""","""Male"""
"""Adelie""","""Torgersen""","""Female"""
"""Adelie""","""Torgersen""","""Female"""
"""Adelie""","""Torgersen""",null
"""Adelie""","""Torgersen""","""Female"""


In [12]:
# exclude_dtype also accepts a list — exclude multiple dtype families at once
(penguins
 .select(exclude_dtype=['datetime', 'object'])
 .head()
)

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""
"""Adelie""","""Torgersen""",null,null,null,null,null
"""Adelie""","""Torgersen""",36.7,19.3,193.0,3450.0,"""Female"""


In [13]:
# Titanic: columns starting with 'p' or 'a', plus all numeric; behaviour here is | and not &
(titanic
 .select(startswith=['p', 'a'], dtype='numeric')
 .head()
)

survived,pclass,age,sibsp,parch,fare,adult_male,alive,alone
i64,i64,f64,i64,i64,f64,bool,str,bool
0,3,22.0,1,0,7.25,true,"""no""",false
1,1,38.0,1,0,71.2833,false,"""yes""",false
1,3,26.0,0,0,7.925,false,"""yes""",true
1,1,35.0,1,0,53.1,false,"""yes""",false
0,3,35.0,0,0,8.05,true,"""no""",true


In [14]:
# endswith — select columns whose name ends with a given suffix
(penguins
 .select(endswith='mm')  # same result as .select(regex='mm$') but more readable
 .head()
)

bill_length_mm,bill_depth_mm,flipper_length_mm
f64,f64,f64
39.1,18.7,181.0
39.5,17.4,186.0
40.3,18.0,195.0
null,null,null
36.7,19.3,193.0


In [15]:
# Columns containing 'bill' or 'flipper' in name, used in a full chain ;  behaviour here is | and not &
(
    penguins
    .filter(species=['Adelie', 'Gentoo'])
    .select('species', 'island', contains=['bill', 'flipper'])
    .by('species', 'island')
    .agg(a=['mean', 'count'])
)


species,island,n,bill_length_mm,bill_depth_mm,flipper_length_mm
str,str,i64,f64,f64,f64
"""Adelie""","""Biscoe""",44,38.975,18.370455,188.795455
"""Adelie""","""Dream""",56,38.501786,18.251786,189.732143
"""Adelie""","""Torgersen""",52,38.95098,18.429412,191.196078
"""Gentoo""","""Biscoe""",124,47.504878,14.982114,217.186992


## Negative selection & exclusion — `-col`, `~col`, negative slices, and `exclude=`
Drop or exclude columns by prefixing names with `-` or `~`, passing a negative slice (`'-start:end'`), or using the `exclude=` parameter. Remaining columns preserve their original relative DataFrame order without needing a separate drop function.

In [16]:
# Drop single column by negating with '-' or '~'
(
    penguins.select('-species')
    .head(3)
)


island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,f64,f64,f64,f64,str
"""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""


In [17]:
# Drop multiple columns
(
    penguins.select('-species', '-island', '-sex')
    .head(3)
)


bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
f64,f64,f64,f64
39.1,18.7,181.0,3750.0
39.5,17.4,186.0,3800.0
40.3,18.0,195.0,3250.0


In [18]:
# Exclude a contiguous slice of columns
(
    penguins.select('-bill_length_mm:body_mass_g')
    .head(3)
)


species,island,sex
str,str,str
"""Adelie""","""Torgersen""","""Male"""
"""Adelie""","""Torgersen""","""Female"""
"""Adelie""","""Torgersen""","""Female"""


In [19]:
# Exclude columns using the exclude= keyword
(
    penguins.select(exclude=['species', 'island'])
    .head(3)
)


bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
f64,f64,f64,f64,str
39.1,18.7,181.0,3750.0,"""Male"""
39.5,17.4,186.0,3800.0,"""Female"""
40.3,18.0,195.0,3250.0,"""Female"""


In [20]:
# Combine positive selection with negative exclusion in a chain
(penguins
 .select(dtype='numeric')
 .select('-bill_depth_mm')
 .head(3)
)

bill_length_mm,flipper_length_mm,body_mass_g
f64,f64,f64
39.1,181.0,3750.0
39.5,186.0,3800.0
40.3,195.0,3250.0


## Columns with spaces — bracket notation `[col]`

Square brackets `[col name]` allow selecting and manipulating columns with spaces without quoting ambiguity, supporting individual names, comma lists, ranges, and exclusions:

In [21]:
# Select columns with spaces using bracket notation
penguins_spaced = penguins.rename(columns={'bill_length_mm': 'bill length mm', 'body_mass_g': 'body mass g'})
(
    penguins_spaced.select('species', '[bill length mm]', '[body mass g]')
    .head()
)


species,bill length mm,body mass g
str,f64,f64
"""Adelie""",39.1,3750.0
"""Adelie""",39.5,3800.0
"""Adelie""",40.3,3250.0
"""Adelie""",null,null
"""Adelie""",36.7,3450.0


In [22]:
# Comma-separated string with bracketed space columns
(
    penguins_spaced.select('species, [bill length mm], [body mass g]')
    .head()
)


species,bill length mm,body mass g
str,f64,f64
"""Adelie""",39.1,3750.0
"""Adelie""",39.5,3800.0
"""Adelie""",40.3,3250.0
"""Adelie""",null,null
"""Adelie""",36.7,3450.0


In [23]:
# Drop a spaced column using negative bracket syntax -[col]
(
    penguins_spaced.select('species', '[bill length mm]', '-[bill length mm]')
    .head()
)


species
str
"""Adelie"""
"""Adelie"""
"""Adelie"""
"""Adelie"""
"""Adelie"""


## Columns with spaces — bracket notation `[col]`

Square brackets `[col name]` allow selecting and manipulating columns with spaces without quoting ambiguity, supporting individual names, comma lists, ranges, and exclusions:

In [24]:
# Select columns with spaces using bracket notation
penguins_spaced = penguins.rename(columns={'bill_length_mm': 'bill length mm', 'body_mass_g': 'body mass g'})
(
    penguins_spaced.select('species', '[bill length mm]', '[body mass g]')
    .head()
)


species,bill length mm,body mass g
str,f64,f64
"""Adelie""",39.1,3750.0
"""Adelie""",39.5,3800.0
"""Adelie""",40.3,3250.0
"""Adelie""",null,null
"""Adelie""",36.7,3450.0


In [25]:
# Comma-separated string with bracketed space columns
(
    penguins_spaced.select('species, [bill length mm], [body mass g]')
    .head()
)


species,bill length mm,body mass g
str,f64,f64
"""Adelie""",39.1,3750.0
"""Adelie""",39.5,3800.0
"""Adelie""",40.3,3250.0
"""Adelie""",null,null
"""Adelie""",36.7,3450.0


In [26]:
# Drop a spaced column using negative bracket syntax -[col]
(
    penguins_spaced.select('species', '[bill length mm]', '-[bill length mm]')
    .head()
)


species
str
"""Adelie"""
"""Adelie"""
"""Adelie"""
"""Adelie"""
"""Adelie"""


## Columns with spaces — bracket notation `[col]`

Square brackets `[col name]` allow selecting and manipulating columns with spaces without quoting ambiguity, supporting individual names, comma lists, ranges, and exclusions:

In [27]:
# Select columns with spaces using bracket notation
penguins_spaced = penguins.rename(columns={'bill_length_mm': 'bill length mm', 'body_mass_g': 'body mass g'})
(
    penguins_spaced.select('species', '[bill length mm]', '[body mass g]')
    .head()
)


species,bill length mm,body mass g
str,f64,f64
"""Adelie""",39.1,3750.0
"""Adelie""",39.5,3800.0
"""Adelie""",40.3,3250.0
"""Adelie""",null,null
"""Adelie""",36.7,3450.0


In [28]:
# Comma-separated string with bracketed space columns
(
    penguins_spaced.select('species, [bill length mm], [body mass g]')
    .head()
)


species,bill length mm,body mass g
str,f64,f64
"""Adelie""",39.1,3750.0
"""Adelie""",39.5,3800.0
"""Adelie""",40.3,3250.0
"""Adelie""",null,null
"""Adelie""",36.7,3450.0


In [29]:
# Drop a spaced column using negative bracket syntax -[col]
(
    penguins_spaced.select('species', '[bill length mm]', '-[bill length mm]')
    .head()
)


species
str
"""Adelie"""
"""Adelie"""
"""Adelie"""
"""Adelie"""
"""Adelie"""
